In [1]:
import os

DATA_PATH = "/kaggle/input/datasets/tysonsiwela/ml-dataset"

assert os.path.exists(DATA_PATH), "❌ DATA_PATH not found"

os.chdir(DATA_PATH)

print("✅ Data:", DATA_PATH)

✅ Data: /kaggle/input/datasets/tysonsiwela/ml-dataset


In [2]:
import shutil
from pathlib import Path

WORK = Path("/kaggle/working")

if WORK.exists():
    shutil.rmtree(WORK, ignore_errors=True)

print("🔥 WORK directory fully wiped")

🔥 WORK directory fully wiped


In [3]:
import shutil
import subprocess
from pathlib import Path
from kaggle_secrets import UserSecretsClient

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
DATA_BASE = Path(DATA_PATH)
REPO_DIR = Path("/kaggle/working/Multi-Bot")

# ─────────────────────────────────────────────
# 1. CLONE OR UPDATE REPO
# ─────────────────────────────────────────────
token = UserSecretsClient().get_secret("GITHUB_TOKEN")
repo_url = f"https://{token}@github.com/AnalystTKZ/Multi-Bot.git"

if REPO_DIR.exists():
    print("🔄 Repo exists → pulling latest")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--rebase"], check=True)
else:
    print("⬇️ Cloning repo...")
    subprocess.run(["git", "clone", repo_url, str(REPO_DIR)], check=True)

print("✅ Repo ready")

# ─────────────────────────────────────────────
# 2. INJECT DATA INTO EXPECTED PATH
# ─────────────────────────────────────────────
TARGET_DATA = REPO_DIR / "trading-system"

print("🔄 Merging dataset into repo (non-destructive)")

for item in DATA_BASE.rglob("*"):
    rel = item.relative_to(DATA_BASE)
    target = TARGET_DATA / rel

    if item.is_dir():
        target.mkdir(parents=True, exist_ok=True)

    else:
        target.parent.mkdir(parents=True, exist_ok=True)

        # Only overwrite if it's actually a data file
        # (optional safeguard — adjust if needed)
        if not target.exists():
            shutil.copy2(item, target)
        else:
            # overwrite ONLY if inside known data folders
            if "processed" in str(target) or "data" in str(target):
                shutil.copy2(item, target)

print("✅ Dataset merged safely into trading-system/")

# ─────────────────────────────────────────────
# 3. (YOU RUN YOUR PIPELINE HERE)
# ─────────────────────────────────────────────
# e.g. subprocess.run(["python", "run_backtest.py"], cwd=REPO_DIR)

print("🚀 Ready to run your system")

⬇️ Cloning repo...


Cloning into '/kaggle/working/Multi-Bot'...


✅ Repo ready
🔄 Merging dataset into repo (non-destructive)
✅ Dataset merged safely into trading-system/
🚀 Ready to run your system


In [4]:
# Install requirements first
!pip install -r {REPO_DIR}/trading-system/requirements.txt --no-deps

# Then install RL stack explicitly
!pip install stable-baselines3==2.3.0 gymnasium==0.29.1

# Optional (only if you actually use it)
!pip install gym-trading-env || echo "gym-trading-env failed, continuing..."

  Using cached ccxt-4.1.63-py2.py3-none-any.whl.metadata (106 kB)
Using cached ccxt-4.1.63-py2.py3-none-any.whl (4.0 MB)
  Attempting uninstall: ccxt
    Found existing installation: ccxt 3.0.59
    Uninstalling ccxt-3.0.59:
      Successfully uninstalled ccxt-3.0.59
  Using cached ccxt-3.0.59-py2.py3-none-any.whl.metadata (111 kB)
Using cached ccxt-3.0.59-py2.py3-none-any.whl (3.6 MB)
  Attempting uninstall: ccxt
    Found existing installation: ccxt 4.1.63
    Uninstalling ccxt-4.1.63:
      Successfully uninstalled ccxt-4.1.63


In [5]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

CUDA available: False
GPU: None


In [6]:
os.chdir("/kaggle/working/Multi-Bot/trading-system/trading-engine")
subprocess.run(["python", "scripts/robustness_backtest.py"], check=False)


2026-04-26 09:03:26,469 WARNING split_summary.json not found — using hardcoded epoch dates
2026-04-26 09:03:26,471 INFO Robustness backtest starting
2026-04-26 09:03:26,471 INFO   Symbols : ['USDZAR', 'USDMXN', 'EURAUD', 'GBPCHF', 'AUDJPY']
2026-04-26 09:03:26,471 INFO   Epochs  : [1, 2, 3]
2026-04-26 09:03:26,471 INFO   Data dir: /kaggle/input/datasets/tysonsiwela/ml-dataset/processed_data/histdata
2026-04-26 09:03:26,509 WARNING RegimeClassifier: CUDA unavailable — using CPU
2026-04-26 09:03:26,515 INFO RegimeClassifier[mode=htf_bias] loaded from /kaggle/working/Multi-Bot/trading-system/trading-engine/weights/regime_htf.pkl (device=cpu, features=34, n_classes=3)
2026-04-26 09:03:26,515 INFO [OK] RegimeClassifier regime_htf loaded
2026-04-26 09:03:26,519 INFO RegimeClassifier[mode=ltf_behaviour] loaded from /kaggle/working/Multi-Bot/trading-system/trading-engine/weights/regime_ltf.pkl (device=cpu, features=18, n_classes=4)
2026-04-26 09:03:26,519 INFO [OK] RegimeClassifier regime_ltf 


  ROBUSTNESS BACKTEST — Out-of-sample symbols (model never trained on these)
  Symbol     Epoch  Period                          Trades    WR%     PF  Return%   MaxDD%  Sharpe   TP1%   TP2%
----------------------------------------------------------------------------------------------------
  USDZAR     1      Pre-val era (2016-2022)             58  63.8%   1.70     4.7%     1.4%   19.83   0.0%  36.2%
----------------------------------------------------------------------------------------------------
  [AVG]      1      Pre-val era (2016-2022)             58  63.8%   1.70     4.7%     1.4%   19.83
  OVERALL    ALL    All epochs / all symbols            58  63.8%   1.70     4.7%     1.4%   19.83


  ROBUSTNESS BACKTEST — Out-of-sample symbols (model never trained on these)
  Symbol     Epoch  Period                          Trades    WR%     PF  Return%   MaxDD%  Sharpe   TP1%   TP2%
----------------------------------------------------------------------------------------------------
  U

2026-04-26 09:10:57,564 INFO   USDMXN epoch 2 done: 18 trades, WR=61.1%, PF=1.12, Return=0.3%
2026-04-26 09:10:57,573 INFO Checkpoint saved (7 combinations done)
2026-04-26 09:10:57,583 WARNING GITHUB_TOKEN not set — skipping GitHub push after USDMXN_epoch2
2026-04-26 09:10:57,654 INFO === EURAUD Epoch 2 (2023-01-01 → 2024-08-27) ===
2026-04-26 09:10:57,655 INFO PM: ml_trader buy EURAUD conf=0.59 R:R 2.00/3.50 size=7607.69 (base=7607.69 vol=1.00 streak=1.00 daily_left=40.00)
2026-04-26 09:10:57,656 INFO PM: ml_trader sell EURAUD conf=0.62 R:R 2.00/3.50 size=7500.65 (base=7500.65 vol=1.00 streak=1.00 daily_left=40.00)
2026-04-26 09:10:57,657 INFO PM: ml_trader sell EURAUD conf=0.60 R:R 2.00/3.50 size=6466.70 (base=6466.70 vol=1.00 streak=1.00 daily_left=40.00)
2026-04-26 09:10:57,658 INFO PM: ml_trader sell EURAUD conf=0.61 R:R 2.00/3.50 size=6778.88 (base=6778.88 vol=1.00 streak=1.00 daily_left=40.00)
2026-04-26 09:10:57,659 INFO PM: ml_trader sell EURAUD conf=0.61 R:R 2.00/3.50 size=7



  ROBUSTNESS BACKTEST — Out-of-sample symbols (model never trained on these)
  Symbol     Epoch  Period                          Trades    WR%     PF  Return%   MaxDD%  Sharpe   TP1%   TP2%
----------------------------------------------------------------------------------------------------
  USDZAR     1      Pre-val era (2016-2022)             58  63.8%   1.70     4.7%     1.4%   19.83   0.0%  36.2%
  USDMXN     1      Pre-val era (2016-2022)             46  58.7%   1.36     2.3%     1.3%   11.84   0.0%  45.6%
  EURAUD     1      Pre-val era (2016-2022)             58  65.5%   1.73     4.7%     1.1%   20.22   0.0%  44.8%
  GBPCHF     1      Pre-val era (2016-2022)             45  55.6%   1.12     0.8%     2.1%    4.49   0.0%  40.0%
  AUDJPY     1      Pre-val era (2016-2022)             35  40.0%   0.50    -3.3%     3.7%  -25.45   0.0%  17.1%
  USDZAR     2      Val era / Round 1 (2023-2024)       11  36.4%   0.77    -0.5%     1.2%   -9.92   0.0%  27.3%
  USDMXN     2      Val era /

2026-04-26 09:10:58,950 INFO PM: ml_trader sell USDZAR conf=0.75 R:R 2.50/4.00 size=633.84 (base=633.84 vol=1.00 streak=1.00 daily_left=40.00)
2026-04-26 09:10:58,951 INFO PM: ml_trader sell USDZAR conf=0.59 R:R 2.00/3.50 size=631.39 (base=631.39 vol=1.00 streak=1.00 daily_left=40.00)
2026-04-26 09:10:58,951 INFO PM: ml_trader sell USDZAR conf=0.60 R:R 2.00/3.50 size=825.87 (base=825.87 vol=1.00 streak=1.00 daily_left=40.00)
2026-04-26 09:10:58,954 INFO PM: ml_trader sell USDZAR conf=0.62 R:R 2.00/3.50 size=953.89 (base=953.89 vol=1.00 streak=1.00 daily_left=40.00)
2026-04-26 09:10:58,954 INFO PM: ml_trader sell USDZAR conf=0.71 R:R 2.14/3.64 size=995.49 (base=995.49 vol=1.00 streak=1.00 daily_left=40.00)
2026-04-26 09:10:58,955 INFO PM: ml_trader sell USDZAR conf=0.59 R:R 2.00/3.50 size=678.52 (base=678.52 vol=1.00 streak=1.00 daily_left=40.00)
2026-04-26 09:10:58,958 INFO PM: ml_trader sell USDZAR conf=0.67 R:R 2.00/3.50 size=622.85 (base=622.85 vol=1.00 streak=1.00 daily_left=40.00)



  ROBUSTNESS BACKTEST — Out-of-sample symbols (model never trained on these)
  Symbol     Epoch  Period                          Trades    WR%     PF  Return%   MaxDD%  Sharpe   TP1%   TP2%
----------------------------------------------------------------------------------------------------
  USDZAR     1      Pre-val era (2016-2022)             58  63.8%   1.70     4.7%     1.4%   19.83   0.0%  36.2%
  USDMXN     1      Pre-val era (2016-2022)             46  58.7%   1.36     2.3%     1.3%   11.84   0.0%  45.6%
  EURAUD     1      Pre-val era (2016-2022)             58  65.5%   1.73     4.7%     1.1%   20.22   0.0%  44.8%
  GBPCHF     1      Pre-val era (2016-2022)             45  55.6%   1.12     0.8%     2.1%    4.49   0.0%  40.0%
  AUDJPY     1      Pre-val era (2016-2022)             35  40.0%   0.50    -3.3%     3.7%  -25.45   0.0%  17.1%
  USDZAR     2      Val era / Round 1 (2023-2024)       11  36.4%   0.77    -0.5%     1.2%   -9.92   0.0%  27.3%
  USDMXN     2      Val era /

2026-04-26 09:11:00,281 INFO PM: ml_trader buy AUDJPY conf=0.74 R:R 2.43/3.93 size=148.52 (base=148.52 vol=1.00 streak=1.00 daily_left=40.00)
2026-04-26 09:11:00,282 INFO PM: ml_trader buy AUDJPY conf=0.69 R:R 2.00/3.50 size=97.35 (base=97.35 vol=1.00 streak=1.00 daily_left=40.00)
2026-04-26 09:11:00,283 INFO PM: ml_trader buy AUDJPY conf=0.61 R:R 2.00/3.50 size=129.82 (base=129.82 vol=1.00 streak=1.00 daily_left=40.00)
2026-04-26 09:11:00,285 INFO PM: ml_trader buy AUDJPY conf=0.65 R:R 2.00/3.50 size=148.92 (base=148.92 vol=1.00 streak=1.00 daily_left=40.00)
2026-04-26 09:11:00,285 INFO PM: ml_trader buy AUDJPY conf=0.61 R:R 2.00/3.50 size=157.64 (base=157.64 vol=1.00 streak=1.00 daily_left=40.00)
2026-04-26 09:11:00,288 INFO PM: ml_trader buy AUDJPY conf=0.61 R:R 2.00/3.50 size=147.98 (base=147.98 vol=1.00 streak=1.00 daily_left=40.00)
2026-04-26 09:11:00,288 INFO PM: ml_trader buy AUDJPY conf=0.60 R:R 2.00/3.50 size=135.56 (base=135.56 vol=1.00 streak=1.00 daily_left=40.00)
2026-04-



  ROBUSTNESS BACKTEST — Out-of-sample symbols (model never trained on these)
  Symbol     Epoch  Period                          Trades    WR%     PF  Return%   MaxDD%  Sharpe   TP1%   TP2%
----------------------------------------------------------------------------------------------------
  USDZAR     1      Pre-val era (2016-2022)             58  63.8%   1.70     4.7%     1.4%   19.83   0.0%  36.2%
  USDMXN     1      Pre-val era (2016-2022)             46  58.7%   1.36     2.3%     1.3%   11.84   0.0%  45.6%
  EURAUD     1      Pre-val era (2016-2022)             58  65.5%   1.73     4.7%     1.1%   20.22   0.0%  44.8%
  GBPCHF     1      Pre-val era (2016-2022)             45  55.6%   1.12     0.8%     2.1%    4.49   0.0%  40.0%
  AUDJPY     1      Pre-val era (2016-2022)             35  40.0%   0.50    -3.3%     3.7%  -25.45   0.0%  17.1%
  USDZAR     2      Val era / Round 1 (2023-2024)       11  36.4%   0.77    -0.5%     1.2%   -9.92   0.0%  27.3%
  USDMXN     2      Val era /

CompletedProcess(args=['python', 'scripts/robustness_backtest.py'], returncode=0)

In [ ]:
%run /kaggle/working/Multi-Bot/trading-system/kaggle_train.py

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
GITHUB_TOKEN = user_secrets.get_secret("GITHUB_TOKEN")

In [ ]:
# ── Push trained weights to GitHub ───────────────────────────────────────────
import os, shutil, subprocess
from pathlib import Path
from datetime import datetime, timezone
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
GITHUB_TOKEN  = secrets.get_secret("GITHUB_TOKEN")
GITHUB_REPO   = os.getenv("GITHUB_REPO",   "AnalystTKZ/Multi-Bot")
GITHUB_BRANCH = os.getenv("GITHUB_BRANCH", "main")
GITHUB_USER   = os.getenv("GITHUB_USER",   "Kaggle Training Bot")
GITHUB_EMAIL  = os.getenv("GITHUB_EMAIL",  "bot@kaggle.local")

REPO_ROOT  = Path("/kaggle/working/Multi-Bot")
REMOTE_TS  = REPO_ROOT / "trading-system"
WORK_TS    = Path("/kaggle/working/Multi-Bot/trading-system")
ENGINE_SRC = WORK_TS   / "trading-engine"
ENGINE_DST = REMOTE_TS / "trading-engine"

def _git(cmd, check=True):
    r = subprocess.run(cmd, cwd=str(REPO_ROOT), capture_output=True, text=True, check=check)
    if r.stdout.strip(): print(r.stdout.strip())
    if r.stderr.strip(): print(r.stderr.strip())
    return r

if not REPO_ROOT.exists():
    raise RuntimeError(f"Git clone not found at {REPO_ROOT}. Run notebook cell 0 first.")

# 1. Auth + pull
remote_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"
_git(["git", "config", "user.name",  GITHUB_USER])
_git(["git", "config", "user.email", GITHUB_EMAIL])
_git(["git", "remote", "set-url", "origin", remote_url])
_git(["git", "pull", "--ff-only", "origin", GITHUB_BRANCH], check=False)

# 2. Copy weights + logs from working copy into clone (no-op if env_config already wrote there)
for subdir in ["weights", "logs", "backtest_results"]:
    src = ENGINE_SRC / subdir
    dst = ENGINE_DST / subdir
    if src.exists():
        dst.mkdir(parents=True, exist_ok=True)
        for f in src.rglob("*"):
            if f.is_file():
                rel   = f.relative_to(src)
                dst_f = dst / rel
                dst_f.parent.mkdir(parents=True, exist_ok=True)
                if not dst_f.exists() or f.stat().st_mtime > dst_f.stat().st_mtime:
                    shutil.copy2(f, dst_f)

for subdir in ["ml_training"]:
    src = WORK_TS / subdir
    dst = REMOTE_TS / subdir
    if src.exists():
        dst.mkdir(parents=True, exist_ok=True)
        for f in src.rglob("*"):
            if f.is_file():
                rel   = f.relative_to(src)
                dst_f = dst / rel
                dst_f.parent.mkdir(parents=True, exist_ok=True)
                if not dst_f.exists() or f.stat().st_mtime > dst_f.stat().st_mtime:
                    shutil.copy2(f, dst_f)

# 3. Stage
_git(["git", "add",
      "trading-system/trading-engine/weights/",
      "trading-system/trading-engine/logs/",
      "trading-system/trading-engine/backtest_results/",
      "trading-system/ml_training/"])

# 4. Commit + push if anything changed
status = _git(["git", "status", "--porcelain"])
if not status.stdout.strip():
    print("Nothing to commit — weights already up to date on GitHub.")
else:
    ts = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
    _git(["git", "commit", "-m", f"chore(weights): trained weights snapshot — {ts}"])
    _git(["git", "push", "origin", f"HEAD:{GITHUB_BRANCH}"])
    print("✓ Weights pushed to GitHub.")


In [ ]:
import subprocess
from pathlib import Path
from kaggle_secrets import UserSecretsClient

REPO_ROOT     = Path("/kaggle/working/Multi-Bot")
GITHUB_TOKEN  = UserSecretsClient().get_secret("GITHUB_TOKEN")
GITHUB_BRANCH = "main"

def _git(cmd, check=False):
    r = subprocess.run(cmd, cwd=str(REPO_ROOT), capture_output=True, text=True, check=check)
    if r.stdout.strip(): print(r.stdout.strip())
    if r.stderr.strip(): print(r.stderr.strip())
    return r

_git(["git", "pull", "--rebase", "origin", GITHUB_BRANCH])
r = _git(["git", "push", "origin", f"HEAD:{GITHUB_BRANCH}"])
if r.returncode == 0:
    print("✓ Pushed.")
else:
    print("✗ Failed — see stderr above.")


In [ ]:
import os
import subprocess
from pathlib import Path
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("GITHUB_TOKEN")

repo_url = f"https://{token}@github.com/AnalystTKZ/Multi-Bot.git"
target_dir = Path("/kaggle/working/Multi-Bot")

subprocess.run(["git", "-C", repo_path, "add", "."], check=True)
subprocess.run(
    ["git", "-C", repo_path, "commit", "-m", "Update before rebase"],
    check=True
)

subprocess.run(
    ["git", "-C", repo_path, "pull", "--rebase"],
    check=True
)

subprocess.run(
    ["git", "-C", repo_path, "push"],
    check=True
)

print("🚀 Done")

print("✅ Synced and pushed successfully")

In [ ]:
import os, subprocess, json, shutil
from kaggle_secrets import UserSecretsClient
from pathlib import Path

# ── Auth ──────────────────────────────────────────────────────────────────────
secrets = UserSecretsClient()
os.environ["KAGGLE_USERNAME"] = "tysonsiwela"
os.environ["KAGGLE_KEY"]      = secrets.get_secret("KAGGLE_API")


SRC        = Path("/kaggle/working/outputs/processed_data")
UPLOAD_DIR = Path("/kaggle/working/dataset_upload")

# Flatten processed_data/ contents into upload dir
if UPLOAD_DIR.exists():
    shutil.rmtree(UPLOAD_DIR)
UPLOAD_DIR.mkdir()

for f in SRC.rglob("*"):
    if f.is_file():
        dst = UPLOAD_DIR / f.relative_to(SRC)
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, dst)
        print(f"  {f.relative_to(SRC)}")

meta = {
    "title": "ml-dataset",
    "id": "tysonsiwela/ml-dataset",
    "licenses": [{"name": "CC0-1.0"}]
}
(UPLOAD_DIR / "dataset-metadata.json").write_text(json.dumps(meta, indent=2))

result = subprocess.run(
    ["kaggle", "datasets", "version", "-p", str(UPLOAD_DIR),
     "-m", "training outputs update", "--dir-mode", "zip"],
)
if result.returncode == 0:
    print("=== Dataset updated: tysonsiwela/ml-dataset ===")
else:
    print("Dataset update FAILED")